# Linux bhyve VM Walkaround

Interactive architectural walkaround for the `linux-python-bhyve` (Linux (Python 3, bhyve)) runtime.


## 0. You Are Here
Execute an immediate runtime identity probe inside this kernel to verify the execution environment.


In [ ]:
%%sh
uname -srm
cat /etc/os-release
printf 'Hypervisor: '
cat /sys/class/dmi/id/product_name 2>/dev/null || true


## 1. Architecture: The Three Planes
The notebook document is hosted by JupyterLab on the host, while code cells execute in the guest runtime. Provisioning occurs on the host, outside this execution environment.

```mermaid
graph LR
    Browser[JupyterLab / Notebook document]
    Server[Jupyter Server<br/>unprivileged host process]
    Provisioner[Kernel Provisioner]
    Daemon[Root runtime daemon]
    Runtime[Selected runtime]
    Kernel[ipykernel]

    Browser --> Server
    Server --> Provisioner
    Provisioner -->|Unix socket| Daemon
    Daemon --> Runtime
    Provisioner -->|SSH + port forwards| Runtime
    Runtime --> Kernel
```


## 2. Kernel Contract
**Security Boundary:** Separate Linux guest kernel and virtual hardware provide a strong hypervisor boundary. The target ABI is Linux, not FreeBSD. The provisioner explicitly asks the runtime daemon for its capabilities and refuses startup unless bhyve.linux is present.

**Constraints & Requirements:**
- **Startup Timeout:** `90s`
- **Networking:** Loopback ZMQ forwarded over SSH; no direct guest port exposure
- **Control Plane:** Privileged operations delegated to root `runtime.sock`
- **Capability Requirement:** `Requires runtime daemon capability: bhyve.linux`


## 3. How It Is Launched
From [`freebsd_laboratory/kernels/linux-python-bhyve/kernel.json`](../freebsd_laboratory/kernels/linux-python-bhyve/kernel.json):

```json
{
  "argv": [
    "/usr/bin/python3",
    "-m",
    "ipykernel_launcher",
    "-f",
    "{connection_file}"
  ],
  "display_name": "Linux (Python 3, bhyve)",
  "language": "python",
  "interrupt_mode": "message",
  "metadata": {
    "kernel_provisioner": {
      "provisioner_name": "linux-bhyve-provisioner",
      "config": {
        "runtime_socket": "/var/run/freebsd-laboratory/runtime.sock",
        "ssh_user": "freebsd",
        "startup_timeout": 90,
        "ssh_connect_timeout": 5,
        "ssh_connection_attempts": 3,
        "ssh_server_alive_interval": 15,
        "ssh_server_alive_count_max": 4
      }
    }
  }
}
```

From [`freebsd_laboratory/bhyve.py`](../freebsd_laboratory/bhyve.py):

```python
def _request_create(
        self,
        name: str,
        owner_pid: int,
        ssh_public_key: str,
    ) -> dict[str, Any]:
        return self._client().create_bhyve(
            name,
            owner_pid,
            ssh_public_key,
            profile="freebsd-python",
        )


class LinuxBhyveProvisioner(RemoteRuntimeProvisioner):
    """Run ipykernel inside an ephemeral Linux bhyve VM on the private lab switch.
```


## 4. Runtime Security Boundary
Separate Linux guest kernel and virtual hardware provide a strong hypervisor boundary. The target ABI is Linux, not FreeBSD. The provisioner explicitly asks the runtime daemon for its capabilities and refuses startup unless bhyve.linux is present.

```mermaid
graph TD
    Host[Host System] -->|SSH / Loopback ZMQ| PF[PF Firewall]
    PF -->|TCP/22 only| Bridge[labbridge0]
    Bridge --> Tap[tap interface]
    Tap --> VM[Linux bhyve VM]
    VM --> Kernel[Python / ipykernel]
```


## 5. Inspect the Runtime
Execute safe, read-only shell observations inside this runtime.


In [ ]:
%%sh
uname -srm
sockstat -4 -l 2>/dev/null || netstat -tln 2>/dev/null
ifconfig 2>/dev/null || ip addr 2>/dev/null


## 6. Interpret the Evidence
In the host management environment (`freebsd-python-host`), the `%ai` magic queries the locally hosted LLM to interpret runtime findings. Inside this isolated guest runtime, direct network connectivity to the host Jupyter server HTTP endpoints is blocked by packet filtering (PF). On the host control plane, an interpretation query is formulated as:

```python
%ai Based on the output above, explain which observations demonstrate the Linux guest execution environment and which observations do NOT prove anything about the host bhyve hypervisor setup.
```


## 7. Bounded Investigation
Autonomous agent tasks (`%%agent` or `freebsd-lab-agent`) are launched and governed on the **FreeBSD host control plane**, where the controller connects to `/var/run/freebsd-laboratory/runtime.sock` to provision disposable runtimes. An autonomous investigation dispatched from the host control plane follows this pattern:

```python
%%agent --mode bhyve --steps 8
Perform a read-only inspection of this Linux bhyve guest runtime.
Determine Linux kernel release, distribution identity, and network interfaces.
```


## 8. What You Cannot See
Explicitly distinguishing guest-observable facts from control-plane facts:

- **Guest-Observable Facts:** `uname`, network interfaces (e.g. `epair`, `tap`, `vnet`), IP addresses, installed packages, guest OS identity.
- **Control-Plane Facts:** Host PF firewall rules, Jupyter server token validation, bhyve/jail creation commands executed by the daemon, host-side lease allocation, loopback port translation, provisioner decisions.

A notebook executing inside this runtime cannot directly prove that the host used `create_bhyve()` or `jail -c`. Those are control-plane facts supported by the implementation excerpts in Section 3.


## 9. Summary & Design Invariants
Core architectural invariants:

1. **Document vs Runtime:** The notebook document is managed by JupyterLab on the host; code cells execute in the isolated runtime.
2. **Transport Security:** Jupyter TCP channels are tunneled exclusively over loopback SSH port forwards.
3. **Privilege Separation:** The unprivileged Jupyter Server delegates privileged lifecycle operations to the root runtime daemon via `/var/run/freebsd-laboratory/runtime.sock`.
4. **Network Policy:** Guest environments cannot observe or alter host-side packet filtering (PF) policies.
5. **Ownership Scoping:** Destructive lifecycle operations (GC/cleanup) are strictly scoped to the authenticated owner's UID.
